# Chapter 7: Hotel Reservation System
*System Design Interview Volume 2*

## TL;DR

A hotel reservation system for a large chain (5,000 hotels, 1M rooms) uses a **relational database** with ACID guarantees. Guests reserve a **room type** (not a specific room), with inventory tracked in a `room_type_inventory` table that supports **10% overbooking**. **Double-booking** is prevented via **idempotent APIs** (client-generated reservation_id as primary key) and **database constraints** or **optimistic locking**. The system uses a **microservice architecture** with Hotel, Rate, Reservation, and Payment services, but pragmatically co-locates reservation and inventory data in the same database to avoid distributed transaction complexity. For high scale, **database sharding** by hotel_id and a **Redis cache** with CDC propagation handle the load.

## Requirements

| Type | Requirement | Detail |
|---|---|---|
| Functional | Show hotel/room details | Hotel and room information pages |
| Functional | Reserve a room | Reserve a room type for date range |
| Functional | Admin panel | Add/remove/update hotel or room info |
| Functional | Overbooking | Allow 10% overbooking |
| Non-functional | High concurrency | Handle peak season traffic spikes |
| Non-functional | Moderate latency | A few seconds acceptable for booking |
| Scale | Hotels/rooms | 5,000 hotels, 1M rooms total |

## Estimation: Reservation Volume

In [ ]:
# Back-of-envelope for hotel reservation system
total_rooms = 1_000_000
occupancy_rate = 0.70
avg_stay_days = 3

daily_reservations = int(total_rooms * occupancy_rate / avg_stay_days)
print(f"Daily reservations: ~{daily_reservations:,}")

seconds_per_day = 86_400
reservation_tps = daily_reservations / seconds_per_day
print(f"Reservation TPS: ~{reservation_tps:.0f}")

# QPS funnel (10% conversion at each step)
detail_page_qps = reservation_tps * 100  # 10x10 funnel
booking_page_qps = reservation_tps * 10
print(f"\nDetail page QPS: ~{detail_page_qps:.0f}")
print(f"Booking page QPS: ~{booking_page_qps:.0f}")
print(f"Reserve TPS: ~{reservation_tps:.0f}")

# Inventory table size
hotels = 5_000
room_types_per_hotel = 20
years_ahead = 2
inventory_rows = hotels * room_types_per_hotel * years_ahead * 365
print(f"\nInventory table rows: {inventory_rows:,}")
print(f"Fits in single DB: {'Yes' if inventory_rows < 100_000_000 else 'No'}")

## High-Level Design

```mermaid
graph TB
    subgraph Clients["Clients"]
        USER[User -- Mobile/Web]
        ADMIN[Admin -- Hotel Staff]
    end

    subgraph Edge["Edge Layer"]
        CDN[CDN -- Static Assets]
        GW["Public API Gateway -- Rate Limiting, Auth"]
    end

    subgraph Services["Microservices"]
        HS[Hotel Service]
        RS[Rate Service]
        RES[Reservation Service]
        PS[Payment Service]
        HMS[Hotel Management Service]
    end

    subgraph Data["Data Layer"]
        HC[Hotel Cache -- Redis]
        HDB[(Hotel DB)]
        RDB[(Rate DB)]
        RESDB[(Reservation + Inventory DB)]
        PDB[(Payment DB)]
    end

    USER --> CDN
    USER --> GW
    ADMIN --> HMS
    GW --> HS
    GW --> RS
    GW --> RES
    RES --> PS
    HS --> HC
    HS --> HDB
    RS --> RDB
    RES --> RESDB
    PS --> PDB
    HMS --> HS
    HMS --> RES
```

## Deep Dive

### Reservation Data Model

The key insight: **guests reserve a room type, not a specific room**. Room numbers are assigned at check-in.

```mermaid
graph LR
    subgraph Tables["Key Tables"]
        RTI["room_type_inventory<br/>---<br/>hotel_id PK<br/>room_type_id PK<br/>date PK<br/>total_inventory<br/>total_reserved"]
        RES["reservation<br/>---<br/>reservation_id PK<br/>hotel_id<br/>room_type_id<br/>start_date<br/>end_date<br/>status<br/>guest_id"]
        RTR["room_type_rate<br/>---<br/>hotel_id PK<br/>room_type_id PK<br/>date PK<br/>rate"]
    end
```

**Reservation check** (SQL logic):
1. SELECT rows from `room_type_inventory` for the date range
2. For each date: verify `total_reserved + rooms_requested <= 110% * total_inventory`
3. If all pass, UPDATE `total_reserved` and INSERT into `reservation`

### Concurrency Control

Three approaches to prevent double-booking when multiple users try to book the last room:

```mermaid
graph TB
    subgraph Pessimistic["Option 1: Pessimistic Locking"]
        P1["SELECT ... FOR UPDATE"]
        P2["Other transactions WAIT"]
        P3["Lock released on COMMIT"]
    end

    subgraph Optimistic["Option 2: Optimistic Locking"]
        O1["Read row + version number"]
        O2["Write with version check"]
        O3["Retry if version mismatch"]
    end

    subgraph DBConstraint["Option 3: Database Constraint"]
        D1["CHECK total_inventory<br/>minus total_reserved >= 0"]
        D2["DB rejects violating UPDATE"]
        D3["No application-level locking"]
    end

    P1 --> P2 --> P3
    O1 --> O2 --> O3
    D1 --> D2 --> D3
```

| Approach | Pros | Cons | Best For |
|---|---|---|---|
| **Pessimistic** | Prevents conflicts entirely | Deadlock risk, not scalable | High contention |
| **Optimistic** | No locks, fast when contention low | Retry storms under high load | Low contention |
| **DB Constraint** | Simplest to implement | Not all DBs support it | Moderate QPS (hotel system) |

**Recommendation**: DB constraints or optimistic locking -- hotel reservation QPS is low enough that retry storms are rare.

### Idempotent API Design

In [ ]:
# Illustrate how idempotency key prevents double-booking
# The reservation_id is generated client-side and used as PK

reservation_request = {
    "reservationID": "13422445",  # idempotency key = primary key
    "hotelID": "245",
    "roomTypeID": "12354673389",
    "startDate": "2021-04-28",
    "endDate": "2021-04-30",
}

# First click: INSERT succeeds
print("First click: INSERT INTO reservation ...")
print(f"  reservation_id = {reservation_request['reservationID']}")
print("  Result: SUCCESS")

# Second click: INSERT fails due to unique constraint on PK
print("\nSecond click: INSERT INTO reservation ...")
print(f"  reservation_id = {reservation_request['reservationID']} (same)")
print("  Result: REJECTED (unique constraint violation)")
print("\nDouble-booking prevented without any application-level locking!")

### Scaling for High Traffic

```mermaid
graph TB
    subgraph Scaling["Scaling Strategy"]
        subgraph Sharding["Database Sharding"]
            SH["Shard by hash of hotel_id"]
            S1["Shard 1"]
            S2["Shard 2"]
            SN["Shard N"]
        end

        subgraph Caching["Redis Cache Layer"]
            RC["Inventory Cache"]
            INV["key: hotelID_roomTypeID_date<br/>value: available rooms"]
        end

        subgraph CDC["Change Data Capture"]
            DB2["Inventory DB -- source of truth"]
            DEB["Debezium"]
            REDIS["Redis Cache"]
        end
    end

    SH --> S1
    SH --> S2
    SH --> SN
    RC --> INV
    DB2 --> DEB --> REDIS
```

**Key principle**: Even with stale cache data, the **database always performs the final validation**. Cache inconsistency only affects UX (user sees availability that turns out to be gone), never correctness.

In [ ]:
# Database sharding calculation
total_qps = 30_000  # booking.com scale
num_shards = 16
qps_per_shard = total_qps / num_shards
print(f"Total QPS: {total_qps:,}")
print(f"Shards: {num_shards}")
print(f"QPS per shard: {qps_per_shard:,.0f}")
print(f"Within MySQL capacity: {'Yes' if qps_per_shard < 5000 else 'No'}")

### Data Consistency in Microservices

In a pure microservice design, cross-service ACID transactions are impossible. Two approaches:

| Approach | How It Works | Trade-off |
|---|---|---|
| **2PC** | Coordinator asks all nodes to prepare, then commit | Blocking -- one slow node stalls everything |
| **Saga** | Chain of local transactions with compensating rollbacks | Eventual consistency -- complex failure handling |

**Pragmatic choice**: Co-locate reservation and inventory in the same database to leverage single-DB ACID. This avoids the complexity of distributed transactions entirely.

## Trade-offs

| Dimension | Option A | Option B |
|---|---|---|
| Relational DB | ACID guarantees, familiar tooling | Harder to scale writes |
| NoSQL | Write-scalable | No ACID, harder concurrency control |
| Pessimistic locking | No conflicts | Deadlocks, poor scalability |
| Optimistic locking | Fast when contention is low | Retry storms under load |
| DB constraints | Simple, DB-level enforcement | Not portable across all DBs |
| Co-located services | Simple ACID transactions | Less microservice purity |
| Pure microservices | Independent scaling/deployment | 2PC/Saga complexity |
| Redis cache | Reduced DB load, fast reads | Cache-DB consistency challenges |
| No cache | Simpler, always consistent | DB becomes bottleneck at scale |

## Key Takeaways

1. **Reserve room types, not specific rooms** -- room numbers are assigned at check-in
2. **Idempotent APIs** with reservation_id as PK prevent double-booking from duplicate clicks
3. **Database constraints** or optimistic locking are sufficient for hotel reservation QPS
4. **ACID matters**: co-locating reservation + inventory in one DB avoids distributed transaction headaches
5. **Overbooking** is trivially implemented: `total_reserved <= 110% * total_inventory`
6. **Cache is an optimization, not the source of truth** -- DB always does final validation
7. **Shard by hotel_id** when scaling to booking.com levels (~30K QPS)

## Related Concepts

- [[reservation-data-model]] -- room_type_inventory table and overbooking logic
- [[concurrency-control]] -- pessimistic, optimistic, and constraint-based locking
- [[idempotent-reservation-api]] -- preventing double-booking via unique keys
- [[hotel-microservice-architecture]] -- service decomposition and pragmatic choices
- [[database-sharding-and-caching]] -- scaling with sharding and Redis
- [[distributed-transactions-saga]] -- 2PC and Saga for cross-service consistency